Ten skrypt ładuje surowe dane wejściowe do chmury Databricks

In [0]:
from threading import Thread
from config import MYSQL_HOST, MYSQL_USER, MYSQL_PASSWORD

jdbc_url = f"jdbc:mysql://{MYSQL_HOST}:3306/wheelie"

def connection():
    return spark.read.format("jdbc") \
        .option("url", jdbc_url) \
        .option("user", MYSQL_USER) \
        .option("password", MYSQL_PASSWORD)

tables = (connection()
    .option("query", "select table_name from information_schema.tables where table_schema = 'wheelie' and table_type <> 'view'")
    .load()
)

table_names = [r.table_name for r in tables.collect()]

tasks = []

for name in table_names:
    task = Thread(target = lambda n=name: connection().option('query', f"select * from {n}").load().write.format("delta").mode("overwrite").saveAsTable(f"landing.{n}"))
    tasks.append(task)
    task.start()

for task in tasks:
    task.join()

kod po polaczeniu z baza, wykonuje zapytanie do meta danych w celu wyszukania wszyskich tabel w bazie.   
Nastepnie wielowatkowo nadpisuje obecne dane w tabelach w schema landing, na koniec synchronizuje wątki by miec pewność, że wszystkie się wykonały przed kolejnym krokiem